# 01 - RMSNorm (AI Infra 视角)

本节从 **工程实现** 角度理解 RMSNorm：
- 为什么用 RMSNorm 而不是 LayerNorm
- 性能对比
- 实现细节
- Kernel Fusion

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)

## 1. 归一化的作用 (30秒版)

**问题**: 深层网络的数值会爆炸或消失

**解决**: 每层输出归一化到稳定范围

```
没有归一化:  x → Linear → 数值爆炸 → Linear → NaN
有归一化:    x → Linear → Norm → 稳定 → Linear → Norm → 稳定
```

In [2]:
# 演示数值爆炸
x = torch.randn(2, 64)
print(f"初始 std: {x.std():.4f}")

for i in range(10):
    W = torch.randn(64, 64) * 1.5
    x = x @ W
    
print(f"10层后 std: {x.std():.4f}  ← 数值爆炸!")

初始 std: 0.9617
10层后 std: 59260047360.0000  ← 数值爆炸!


## 2. LayerNorm vs RMSNorm

| | LayerNorm | RMSNorm |
|--|-----------|--------|
| **公式** | `(x - mean) / std * γ + β` | `x / RMS(x) * γ` |
| **计算** | 2次遍历 (求mean, 求var) | 1次遍历 (求x²均值) |
| **参数** | γ, β | 通常只有 γ |
| **去中心化** | 有 | 无 |

### RMSNorm 的核心公式

```
RMS(x) = sqrt(mean(x²))
RMSNorm(x) = x / RMS(x)
```

In [3]:
# RMSNorm 实现
def rms_norm(x, eps=1e-6):
    """RMSNorm: x / sqrt(mean(x²))"""
    rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    return x / rms

# 测试
x = torch.randn(2, 4)
y = rms_norm(x)

print(f"输入 RMS: {x.pow(2).mean(dim=-1).sqrt()}")
print(f"输出 RMS: {y.pow(2).mean(dim=-1).sqrt()}  ← 接近 1")

输入 RMS: tensor([0.3492, 1.5616])
输出 RMS: tensor([1.0000, 1.0000])  ← 接近 1


## 3. 为什么 RMSNorm 可以替代 LayerNorm?

**论文发现**: LayerNorm 的主要作用是 **缩放**，去中心化的贡献很小

实验证明:
- 在 Transformer 上效果相当
- 计算更少 → 训练更快

### nanochat 更激进: 连 γ 都去掉了!

```python
# nanochat/gpt.py
def norm(x):
    return F.rms_norm(x, (x.size(-1),))  # 无可学习参数
```

## 4. 性能对比

### 计算量分析

In [4]:
# LayerNorm 操作数
def count_layernorm_ops(n):
    """n 个元素的 LayerNorm 操作数"""
    # 1. 求 mean: n 次加法 + 1 次除法
    # 2. x - mean: n 次减法
    # 3. (x-mean)²: n 次乘法
    # 4. 求 var: n 次加法 + 1 次除法
    # 5. sqrt(var): 1 次
    # 6. (x-mean)/std: n 次除法
    # 7. * γ + β: 2n 次
    return 6*n + 3

# RMSNorm 操作数
def count_rmsnorm_ops(n):
    """n 个元素的 RMSNorm 操作数"""
    # 1. x²: n 次乘法
    # 2. 求 mean: n 次加法 + 1 次除法
    # 3. sqrt: 1 次
    # 4. x/rms: n 次除法
    # 5. * γ: n 次 (可选)
    return 3*n + 2

n = 4096  # 典型 hidden_dim
print(f"LayerNorm: ~{count_layernorm_ops(n)} ops")
print(f"RMSNorm:   ~{count_rmsnorm_ops(n)} ops")
print(f"减少: ~{(1 - count_rmsnorm_ops(n)/count_layernorm_ops(n))*100:.0f}%")

LayerNorm: ~24579 ops
RMSNorm:   ~12290 ops
减少: ~50%


### 实际性能测试

In [7]:
import time

# 准备数据
x = torch.randn(32, 2048, 4096)  # 典型 LLM shape
if torch.cuda.is_available():
    x = x.cuda()

layer_norm = nn.LayerNorm(4096)
if torch.cuda.is_available():
    layer_norm = layer_norm.cuda()

# Warmup
for _ in range(10):
    _ = layer_norm(x)
    _ = F.rms_norm(x, (x.size(-1),))

if torch.cuda.is_available():
    torch.cuda.synchronize()

# Benchmark LayerNorm
start = time.time()
for _ in range(100):
    _ = layer_norm(x)
if torch.cuda.is_available():
    torch.cuda.synchronize()
ln_time = time.time() - start

# Benchmark RMSNorm
start = time.time()
for _ in range(100):
    _ = F.rms_norm(x, (x.size(-1),))
if torch.cuda.is_available():
    torch.cuda.synchronize()
rms_time = time.time() - start

print(f"LayerNorm: {ln_time*1000:.1f} ms")
print(f"RMSNorm:   {rms_time*1000:.1f} ms")
print(f"RMSNorm 快 {ln_time/rms_time:.2f}x")

KeyboardInterrupt: 

## 5. Kernel Fusion

在实际实现中，RMSNorm 通常会和其他操作融合:

```
未融合:                     融合后:
x → RMSNorm → memory        x → [RMSNorm + Linear] → memory
    ↓ 写回显存                         ↓
memory → Linear → memory              一次 kernel

减少显存读写，提升性能
```

### PyTorch 的 torch.compile 自动融合

In [ ]:
# torch.compile 会自动融合 RMSNorm
class Block(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.linear = nn.Linear(dim, dim)
    
    def forward(self, x):
        x = F.rms_norm(x, (x.size(-1),))  # 会被融合
        x = self.linear(x)
        return x

# 编译后，RMSNorm 和 Linear 可能被融合成一个 kernel
# block_compiled = torch.compile(Block(64))

## 6. Pre-Norm vs Post-Norm

RMSNorm 放在哪里？现代 LLM 都用 **Pre-Norm**:

```
Post-Norm (原版 Transformer):    Pre-Norm (现代 LLM):

x → Attn → Add → Norm            x ───────→ Add
      ↑      ↓                          ↑
      └──────┘                   Norm → Attn

问题: Norm 在主路径上               优点: 主路径无 Norm
      阻断梯度直通                       梯度可以直通
```

**Pre-Norm 优势**: 训练更稳定，可以用更大的学习率

## 7. 面试常见问题

### Q1: RMSNorm 和 LayerNorm 的区别?

**答**:
- LayerNorm: 减均值、除标准差
- RMSNorm: 只除 RMS (均方根)，不去中心化
- RMSNorm 更快 (~30%)，效果相当

---

### Q2: 为什么 LLM 都用 Pre-Norm?

**答**:
- Pre-Norm 让残差路径上没有 Norm 阻断
- 梯度可以直通，训练更稳定
- 可以训练更深的网络

---

### Q3: RMSNorm 的 eps 参数是干什么的?

**答**:
- 防止除零: `x / sqrt(mean(x²) + eps)`
- 当 x 全为 0 或很小时，确保数值稳定
- 典型值: 1e-5 或 1e-6

---

### Q4: nanochat 为什么不用可学习的 γ 参数?

**答**:
- 实验表明 γ 的贡献很小
- 减少参数量
- 简化实现
- 后面有 Linear 层可以学习缩放

---

### Q5: BatchNorm 为什么不适合 LLM?

**答**:
- BatchNorm 依赖 batch 统计量，batch size 小时不稳定
- 序列长度变化时，统计量不一致
- 推理时单条数据无法用 batch 统计
- LayerNorm/RMSNorm 是逐样本的，没有这些问题

## 8. 总结速查表

| 主题 | 要点 |
|------|------|
| **RMSNorm 公式** | `x / sqrt(mean(x²) + eps)` |
| **vs LayerNorm** | 不去中心化，快 ~30% |
| **Pre-Norm** | Norm 在残差分支内，梯度直通 |
| **nanochat** | 无可学习参数: `F.rms_norm(x, (x.size(-1),))` |
| **Kernel Fusion** | 常和后续 Linear 融合 |

### 一行代码

```python
# PyTorch 内置 RMSNorm
y = F.rms_norm(x, (x.size(-1),))
```